# **🧠 KBS Project: Advanced Medical Expert System for Hypertension**
### *"Intelligent Clinical Decision Support System (CDSS) for Cardiovascular Risk Assessment"*

---
**Course:** AIE212 - Knowledge-Based Systems  
**Technology Stack:** Python, Semantic Web (RDFlib), Forward Chaining Inference  
**Target Disease:** Hypertension (Group 4)

## 📑 **Comprehensive Project Documentation**

### **1. 📝 Executive Summary & Introduction**
This project implements a **Knowledge-Based System (KBS)** tailored for the diagnosis and severity classification of **Hypertension**. Hypertension, often termed the "silent killer," requires precise monitoring of Blood Pressure (BP) alongside clinical symptoms to prevent life-threatening events like strokes or hypertensive emergencies.

*   **Objective:** To assist medical practitioners and patients in identifying the urgency of care required based on 18 clinical data points.
*   **Primary Goal:** Accurate classification into **Normal, Elevated, Urgent (Hypertensive Urgency),** or **Emergency Referral (Hypertensive Emergency).**

---

### **2. 🧪 Knowledge Acquisition & Modeling**
The domain knowledge was extracted from the **American Heart Association (AHA)** and **European Society of Cardiology (ESC)** guidelines. 

#### **A. Knowledge Representation (RDF)**
The system utilizes the **Resource Description Framework (RDF)** to represent patient data as semantic triples (`Subject` -> `Predicate` -> `Object`). This allows for a flexible and extensible data structure.

#### **B. Quantitative Breakdown:**
*   **Named Facts (18):** Encompasses systolic/diastolic thresholds, end-organ damage symptoms (blurred vision, chest pain), and critical risk factors (pregnancy, age, history).
*   **Explicit Rules (15):** A robust set of logical IF-THEN statements that simulate the reasoning of a cardiologist.

---

### **3. 🏗️ System Architecture & Logic Flow**
The system is built on a modular architecture:

1.  **Fact Base (Semantic Memory):** Initializes the patient profile and populates it with user-provided responses.
2.  **Rule Base (Knowledge Repository):** Contains the medical heuristics in a format compatible with RDF-based querying.
3.  **Inference Engine (Forward Chaining):** 
    *   The engine starts with known facts (e.g., BP > 180/120).
    *   It applies rules to derive intermediate statuses (e.g., Crisis Status).
    *   It continues until it reaches a final diagnosis or urgency level.
4.  **Explanation Facility (Transparency):** Every decision is justified by listing the sequence of rules fired, ensuring clinical accountability.

---

### **4. 📊 Technical Requirements Coverage**
| Requirement | Status | Implementation Detail |
| :--- | :---: | :--- |
| **Named Facts** | ✅ | 18 clinical facts asked via emoji-enhanced interface. |
| **IF-THEN Rules** | ✅ | 15 advanced rules covering comorbidities and emergencies. |
| **Inference Engine** | ✅ | Custom forward-chaining logic using `rdflib.Graph`. |
| **Output Classes** | ✅ | 4 levels: Routine, Clinic Review, Urgent, Emergency. |
| **Explanation Facility**| ✅ | Step-by-step audit trail of fired rules. |
| **User Interface** | ✅ | Interactive command-line (Notebook) + Streamlit Web App. |

---

## 1️⃣ Required Libraries

In [1]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS

## 2️⃣ Setup & Knowledge Base

### Define Namespaces

In [2]:
EX = Namespace('http://medical.org/hypertension/')
P = Namespace('http://medical.org/patient/')

### Define Graph

In [3]:
g = Graph()
g.bind('ex', EX)
g.bind('rdfs', RDFS)

g.add((EX.EmergencySymptom, RDFS.subClassOf, EX.Symptom))
g.add((EX.RiskFactor, RDFS.subClassOf, EX.PatientProfile))

g.add((EX.chest_pain, RDF.type, EX.EmergencySymptom))
g.add((EX.blurred_vision, RDF.type, EX.EmergencySymptom))
g.add((EX.confusion, RDF.type, EX.EmergencySymptom))
g.add((EX.shortness_of_breath, RDF.type, EX.EmergencySymptom))

g.add((EX.pregnancy, RDF.type, EX.RiskFactor))
g.add((EX.heart_disease_history, RDF.type, EX.RiskFactor))
g.add((EX.smoker, RDF.type, EX.RiskFactor))

print("✅ Knowledge Base Initialized Successfully.")

✅ Knowledge Base Initialized Successfully.


## 3️⃣ Logic & Inference Engine

### Print Function 

In [4]:
def print_fancy_header(text, icon="🧠"):
    print(f"\n{icon} {'='*65}")
    print(f"   {text.upper()}")
    print(f"   {'='*65}")

### Main Functions

In [ ]:
def run_inference(patient_uri):
    inferred_triples = []
    fired_rules = []
    
    if (patient_uri, EX.hasCondition, EX.high_bp) in g:
        inferred_triples.append((patient_uri, EX.status, EX.ElevatedStatus))
        fired_rules.append("Rule 1: High BP detected.")

    if (patient_uri, EX.hasCondition, EX.crisis_bp) in g:
        inferred_triples.append((patient_uri, EX.status, EX.CrisisStatus))
        fired_rules.append("Rule 2: Crisis BP detected.")

    for s, p, o in g.triples((None, RDF.type, EX.EmergencySymptom)):
        if (patient_uri, EX.hasCondition, o) in g and (patient_uri, EX.status, EX.CrisisStatus) in inferred_triples:
            inferred_triples.append((patient_uri, EX.urgency, EX.EmergencyReferral))
            fired_rules.append(f"Rule 3: Crisis BP + {o.split('/')[-1]} -> Emergency Referral.")
            break

    if (patient_uri, EX.status, EX.CrisisStatus) in inferred_triples and (patient_uri, EX.urgency, EX.EmergencyReferral) not in inferred_triples:
        inferred_triples.append((patient_uri, EX.urgency, EX.UrgentReview))
        fired_rules.append("Rule 4: Crisis BP -> Hypertensive Urgency.")

    if (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples and (patient_uri, EX.hasCondition, EX.pregnancy) in g:
        inferred_triples.append((patient_uri, EX.urgency, EX.UrgentReview))
        fired_rules.append("Rule 5: High BP + Pregnancy -> Preeclampsia Risk.")

    if (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples and (patient_uri, EX.hasCondition, EX.heart_disease_history) in g:
        inferred_triples.append((patient_uri, EX.urgency, EX.UrgentReview))
        fired_rules.append("Rule 6: High BP + Heart History -> Urgent Review.")

    if (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples and (patient_uri, EX.hasCondition, EX.headache) in g:
        inferred_triples.append((patient_uri, EX.urgency, EX.UrgentReview))
        fired_rules.append("Rule 7: High BP + Severe Headache -> Potential Urgency.")

    if (patient_uri, EX.status, EX.ElevatedStatus) not in inferred_triples and (patient_uri, EX.hasCondition, EX.headache) in g:
        inferred_triples.append((patient_uri, EX.urgency, EX.RoutineCheck))
        fired_rules.append("Rule 8: Symptoms with Normal BP -> Clinic Review.")

    if (patient_uri, EX.hasCondition, EX.smoker) in g and (patient_uri, EX.hasCondition, EX.obesity) in g:
        inferred_triples.append((patient_uri, EX.profile, EX.HighRiskLifestyle))
        fired_rules.append("Rule 9: Combined Lifestyle Risks detected.")

    if (patient_uri, EX.hasCondition, EX.age_over_65) in g and (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples:
        inferred_triples.append((patient_uri, EX.urgency, EX.UrgentReview))
        fired_rules.append("Rule 10: Elderly patient (>65) with high BP.")

    if (patient_uri, EX.urgency, EX.EmergencyReferral) in inferred_triples or (patient_uri, EX.status, EX.CrisisStatus) in inferred_triples:
        inferred_triples.append((patient_uri, EX.suspicion, EX.Probable))
        fired_rules.append("Rule 11: High severity -> Suspicion: PROBABLE.")

    if (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples and (patient_uri, EX.suspicion, EX.Probable) not in inferred_triples:
        inferred_triples.append((patient_uri, EX.suspicion, EX.Possible))
        fired_rules.append("Rule 12: Elevated BP -> Suspicion: POSSIBLE.")

    if (patient_uri, EX.status, EX.ElevatedStatus) not in inferred_triples and (patient_uri, EX.profile, EX.HighRiskLifestyle) not in inferred_triples:
        inferred_triples.append((patient_uri, EX.suspicion, EX.Unlikely))
        fired_rules.append("Rule 13: Normal status -> Suspicion: UNLIKELY.")

    if (patient_uri, EX.urgency, EX.EmergencyReferral) in inferred_triples:
        inferred_triples.append((patient_uri, RDF.type, EX.HypertensiveEmergency))
        fired_rules.append("Rule 14: Confirmed Hypertensive Emergency.")

    if (patient_uri, EX.hasCondition, EX.diabetes) in g and (patient_uri, EX.status, EX.ElevatedStatus) in inferred_triples:
        inferred_triples.append((patient_uri, EX.advice, EX.DiabetesWarning))
        fired_rules.append("Rule 15: High BP + Diabetes -> Increased Risk Warning.")

    for t in inferred_triples: g.add(t)
    return fired_rules

In [ ]:
def evaluate_patient(responses, patient_name="Patient_01"):
    patient_uri = P[patient_name]
    g.remove((patient_uri, None, None))
    
    fact_mapping = {
        "1": EX.high_bp, "2": EX.crisis_bp, "3": EX.headache, "4": EX.dizziness, 
        "5": EX.blurred_vision, "6": EX.chest_pain, "7": EX.shortness_of_breath, 
        "8": EX.confusion, "9": EX.nosebleed, "10": EX.pregnancy, 
        "11": EX.heart_disease_history, "12": EX.smoker, "13": EX.obesity, 
        "14": EX.age_over_65, "15": EX.high_salt_intake, "16": EX.diabetes, 
        "17": EX.sedentary_lifestyle, "18": EX.alcohol_consumption
    }
    
    for key, val in responses.items():
        if val == 'y': g.add((patient_uri, EX.hasCondition, fact_mapping[key]))
    
    rules_fired = run_inference(patient_uri)
    
    susp = "UNLIKELY ⚪"
    if (patient_uri, EX.suspicion, EX.Probable) in g: susp = "PROBABLE 🔴"
    elif (patient_uri, EX.suspicion, EX.Possible) in g: susp = "POSSIBLE 🟡"
    
    urg = "ROUTINE ✅"
    if (patient_uri, EX.urgency, EX.EmergencyReferral) in g: urg = "EMERGENCY REFERRAL 🚨"
    elif (patient_uri, EX.urgency, EX.UrgentReview) in g: urg = "URGENT REVIEW ⚠️"
    elif (patient_uri, EX.urgency, EX.RoutineCheck) in g: urg = "CLINIC REVIEW 🩺"

    return susp, urg, rules_fired

## 4️⃣ Main Application

In [6]:
print_fancy_header("Hypertension Expert System Interface", "🩺")
print("Please answer with 'y' or 'n':\n")

questions = {
    "1": "🩸 Is your Blood Pressure high (>140/90)?", 
    "2": "⚠️ Is your BP reading in a crisis range (>180/120)?",
    "3": "🤕 Do you have a severe headache?", 
    "4": "😵 Are you feeling dizzy?", 
    "5": "👁️ Are you experiencing blurred vision?", 
    "6": "💔 Do you have chest pain?", 
    "7": "🫁 Do you have shortness of breath?", 
    "8": "🧠 Are you feeling confused?",
    "9": "🩸 Do you have a nosebleed?", 
    "10": "🤰 Are you pregnant?", 
    "11": "🏥 History of heart disease?", 
    "12": "🚬 Do you smoke?", 
    "13": "⚖️ Obesity (BMI > 30)?", 
    "14": "👴 Age over 65?",
    "15": "🧂 High salt intake?", 
    "16": "🍬 Do you have Diabetes?", 
    "17": " Couch Potato? (Sedentary lifestyle)?", 
    "18": "🍺 Regular alcohol use?"
}

user_responses = {}
for key, q in questions.items():
    ans = input(f"[?] {q} (y/n): ").strip().lower()
    user_responses[key] = ans

susp, urg, rules_fired = evaluate_patient(user_responses, "User_Patient")

print_fancy_header("Diagnostic Assessment Results", "📊")
print(f"   ▶  Suspicion Level : {susp}")
print(f"   ▶  Urgency Level   : {urg}")

print("\n🔍 EXPLANATION FACILITY (Reasoning Path):")
if not rules_fired:
    print("   - No specific high-risk rules were triggered.")
else:
    for i, r in enumerate(rules_fired):
        print(f"   {i+1}. {r}")
print("="*69)


🩺 =================================================================
   HYPERTENSION EXPERT SYSTEM INTERFACE
Please answer with 'y' or 'n':


📊 =================================================================
   DIAGNOSTIC ASSESSMENT RESULTS
   ▶  Suspicion Level : PROBABLE 🔴
   ▶  Urgency Level   : URGENT REVIEW ⚠️

🔍 EXPLANATION FACILITY (Reasoning Path):
   1. Rule 1: High BP detected.
   2. Rule 2: Crisis BP detected.
   3. Rule 4: Crisis BP -> Hypertensive Urgency.
   4. Rule 5: High BP + Pregnancy -> Preeclampsia Risk.
   5. Rule 6: High BP + Heart History -> Urgent Review.
   6. Rule 7: High BP + Severe Headache -> Potential Urgency.
   7. Rule 9: Combined Lifestyle Risks detected.
   8. Rule 10: Elderly patient (>65) with high BP.
   9. Rule 11: High severity -> Suspicion: PROBABLE.
   10. Rule 15: High BP + Diabetes -> Increased Risk Warning.


## **Thank You🎀🫶🏻💓**